In [0]:
CREATE TABLE IF NOT EXISTS sac.support.ticket (
		ticket_id STRING NOT NULL,
		customer_id STRING,
		timestamp_created TIMESTAMP,
		timestamp_closed TIMESTAMP,
		subject STRING,
		description STRING,
		priority STRING,
		status STRING,
		CONSTRAINT ticket_pk PRIMARY KEY (ticket_id),
		CONSTRAINT ticket_f1 FOREIGN KEY (customer_id) REFERENCES sac.customer.customer (customer_id)
	);

-- Edit constraints of table
ALTER TABLE
	sac.support.ticket
DROP CONSTRAINT IF EXISTS
	support_ticket_ci;

ALTER TABLE
	sac.support.ticket
DROP CONSTRAINT IF EXISTS
	support_ticket_tc;

ALTER TABLE
	sac.support.ticket
DROP CONSTRAINT IF EXISTS
	support_ticket_tl;

ALTER TABLE
	sac.support.ticket
ADD
	CONSTRAINT support_ticket_ci CHECK (customer_id IS NOT NULL);

ALTER TABLE
	sac.support.ticket
ADD
	CONSTRAINT support_ticket_tc
		CHECK (
			(
				status = 'open'
				AND timestamp_closed IS NULL
			)
			OR (
				status != 'open'
				AND timestamp_created <= timestamp_closed
			)
		);

ALTER TABLE
	sac.support.ticket
ADD
	CONSTRAINT support_ticket_tl
		CHECK (
			timestamp_created <= current_date()
			OR (
				timestamp_closed <= current_date()
				AND status != 'open'
			)
		);

-- Add values to table
MERGE INTO
	sac.support.ticket s
USING (
	SELECT
		t.ticket_id,
		t.customer_id,
		CAST(t.timestamp_created AS TIMESTAMP) AS timestamp_created,
		CASE
			WHEN t.timestamp_closed = 'nan' THEN NULL
			ELSE CAST(t.timestamp_closed AS TIMESTAMP)
		END AS timestamp_closed,
		t.subject,
		t.description,
		t.priority,
		t.status
	FROM
		sac.support.ticket_bronze t
	QUALIFY
		row_number() OVER (PARTITION BY t.ticket_id ORDER BY t.ingestion_time DESC) = 1
) b
ON
	s.ticket_id = b.ticket_id
WHEN MATCHED AND
	sha1(
		concat_ws(
			'|',
			b.timestamp_created,
			b.timestamp_closed,
			b.subject,
			b.description,
			b.priority,
			b.status
		)
	)
		!= sha1(
			concat_ws(
				'|',
				s.timestamp_created,
				s.timestamp_closed,
				s.subject,
				s.description,
				s.priority,
				s.status
			)
		)
	THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;